<a href="https://colab.research.google.com/github/Wizard-i/FlyRank-Starter-Repo/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wizard-i/FlyRank-Starter-Repo/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

## 1. My lane: Refresh / Content Opportunity Scoring

I provisionally choose Lane 2: Refresh / Content Opportunity Scoring.

I selected this lane because I want to identify which website pages deserve attention first based on their search visibility, content freshness, and engagement performance. The starter dataset provides observable signals such as impressions, clicks, CTR, average position, content age, and engagement. These signals can help identify potential content opportunities and organize them into a ranked review queue. I will begin with a transparent rule-based baseline and investigate whether machine learning can improve the ranking. My goal is to support content editors in making better review decisions, not to guarantee that updating a page will improve its performance.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "Wizard-i/FlyRank-Starter-Repo/main/"
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

# Check required columns
required_columns = [
    "content_id",
    "client_id",
    "impressions_90d",
    "trend_direction",
    "days_since_last_update",
    "avg_position",
    "ctr"
]

missing_columns = set(required_columns) - set(df.columns)

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

print("Dataset loaded successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

## 2. The question: decision, action, cost of a wrong call

**Research Question:**

Which content pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring based on their observable search and engagement signals?

**Unit of Analysis:**

One pseudonymized content page. Each row in the starter dataset represents one content item.

**Decision:**

The main decision is to determine which content pages should receive editorial attention first, considering limited time and resources.

**Who Takes Action:**

SEO specialists, content editors, and digital marketing teams.

**Possible Actions:**

- Refresh outdated content to improve its usefulness.
- Expand pages that may have insufficient content depth.
- Protect pages that show strong search visibility and performance.
- Investigate pages with declining search visibility.
- Prune or consolidate pages only after checking related content.
- Monitor pages where the available evidence is not strong enough for immediate action.

**Expected Output:**

A ranked content review queue containing a priority score, suggested action, reason codes, and a confidence label for each page.

**Cost of a Wrong Recommendation:**

A false positive may waste editorial time and lead to unnecessary content changes. A false negative may cause a valuable content opportunity or an important performance issue to be overlooked. An incorrect pruning recommendation may also remove useful content.

**Why Data and Machine Learning Can Help:**

Content performance depends on several interacting signals, including impressions, clicks, CTR, average position, freshness, and engagement. A simple rule-based approach can identify obvious opportunities, but it may overlook combinations of signals. I will first establish a transparent baseline and then test whether machine learning provides a measurable improvement in ranking quality. If a simple rule performs sufficiently well, additional model complexity may not be necessary.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify the unit of analysis

print("Total dataset rows:", len(df))
print("Unique content pages:", df["content_id"].nunique())
print("Unique clients:", df["client_id"].nunique())
print("Duplicate content IDs:", df["content_id"].duplicated().sum())

assert df["content_id"].is_unique, (
    "Expected one row per content page."
)

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Explore content opportunities in the starter dataset
numeric_columns = [
    "impressions_90d",
    "days_since_last_update",
    "avg_position",
    "ctr"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 1. Pages with a declining current-window trend
declining = (
    df["trend_direction"]
    .astype("string")
    .str.lower()
    .eq("down")
)

# 2. Old but visible pages
stale_visible = (
    (df["days_since_last_update"] >= 180) &
    (df["impressions_90d"] >= 500)
)

# 3. Visible pages with low CTR
low_ctr_visible = (
    (df["impressions_90d"] >= 500) &
    (df["avg_position"].gt(0)) &
    (df["avg_position"] <= 20) &
    (df["ctr"] < 0.5)
)

print("STARTER DATASET EVIDENCE")
print("=" * 45)

print(
    f"Pages with declining trend: "
    f"{declining.sum():,} "
    f"({declining.mean() * 100:.2f}%)"
)

print(
    f"Stale but visible pages: "
    f"{stale_visible.sum():,}"
)

print(
    f"Low-CTR visible pages: "
    f"{low_ctr_visible.sum():,}"
)

## 4. Careful words: what I can and can't claim

**What I Can Claim:**

- The starter dataset contains observable patterns in content freshness, search visibility, and engagement.
- Some content pages meet measurable criteria that make them reasonable candidates for further review.
- Combining multiple performance signals may help prioritize editorial attention.
- A transparent baseline and machine learning model can be compared using ranking evaluation metrics.
- The final system can provide decision-support recommendations with explainable reason codes.

**What I Cannot Claim:**

- I cannot claim that updating a page will automatically improve its search performance.
- I cannot prove that a particular content signal causes a decline in search visibility.
- I cannot claim to predict Google's ranking algorithm.
- I cannot treat the current trend direction as proof of future performance.
- I cannot assume that every page selected by the ranking system requires an update.

The starter dataset provides a snapshot of observable content and search performance. Its trend_direction field represents a comparison between recent and previous 30-day impressions, not a future outcome.

Therefore, I will treat the initial findings as observational evidence and decision-support signals. Stronger future predictions will require properly defined time-based outcomes and validation using the warehouse dataset.



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check important data limitations

product_decision_columns = [
    "health_score",
    "priority_score",
    "action_type",
    "refresh_tier"
]

present_columns = [
    col for col in product_decision_columns
    if col in df.columns
]

print("Product decision columns found:", present_columns)

print(
    "Current trend field available:",
    "trend_direction" in df.columns
)

print(
    "Future outcome directly available:",
    False
)

print("\nImportant:")
print(
    "The current trend is observational, "
    "not proof of future performance."
)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.